In [0]:
USE CATALOG workspace;
USE SCHEMA causal_project;

In [0]:
WITH ordered_campaigns AS 
(SELECT b.household_key, b.campaign, b.DESCRIPTION, d.START_DAY,d.END_DAY,
row_number() OVER(PARTITION BY b.household_key ORDER BY d.START_DAY) AS rn 
FROM bronze_campaigns b JOIN bronze_campaign_desc d ON b.CAMPAIGN = d.CAMPAIGN
WHERE household_key in (SELECT household_key FROM bronze_demographics)),
first_only AS (SELECT * FROM ordered_campaigns WHERE rn = 1),
contamination AS (SELECT DISTINCT(first.household_key) FROM first_only first 
JOIN bronze_campaigns other on other.household_key= first.household_key AND other.campaign <> first.campaign JOIN bronze_campaign_desc other_d on other_d.CAMPAIGN = other.campaign AND other_d.START_DAY <= first.END_DAY)
SELECT COUNT(*) FROM contamination